# Lab 6 — Silver: enriquecimento com contexto

## Objetivo

Este laboratório constrói a camada Silver a partir das bases Bronze.

As transações serão enriquecidas com o segmento e o credit score dos clientes. Também serão criadas variáveis temporais e uma faixa de valor, facilitando as análises posteriores.

Antes e depois do JOIN, serão realizados controles para identificar transações sem cliente correspondente e verificar se a junção provocou perda ou multiplicação indevida de registros.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "bronze").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "bronze").exists():
            pasta_projeto = pasta_pai
            break

arquivo_clientes_bronze = (
    pasta_projeto / "dados" / "bronze" / "customers.parquet"
)

arquivo_transacoes_bronze = (
    pasta_projeto / "dados" / "bronze" / "transactions.parquet"
)

arquivo_silver = (
    pasta_projeto
    / "dados"
    / "silver"
    / "transactions_enriched.parquet"
)

arquivo_banco = (
    pasta_projeto
    / "dia2_transformacao"
    / "lab06_silver"
    / "silver.duckdb"
)

assert arquivo_clientes_bronze.exists(), (
    "Bronze de clientes não encontrada."
)

assert arquivo_transacoes_bronze.exists(), (
    "Bronze de transações não encontrada."
)

print("Clientes Bronze:", arquivo_clientes_bronze)
print("Transações Bronze:", arquivo_transacoes_bronze)
print("Destino Silver:", arquivo_silver)

Clientes Bronze: C:\BigData\bigdata-curso-gabriel\dados\bronze\customers.parquet
Transações Bronze: C:\BigData\bigdata-curso-gabriel\dados\bronze\transactions.parquet
Destino Silver: C:\BigData\bigdata-curso-gabriel\dados\silver\transactions_enriched.parquet


In [2]:
conexao = duckdb.connect(str(arquivo_banco))

caminho_clientes = (
    arquivo_clientes_bronze.as_posix().replace("'", "''")
)

caminho_transacoes = (
    arquivo_transacoes_bronze.as_posix().replace("'", "''")
)

conexao.execute(f"""
    CREATE OR REPLACE TABLE bronze_customers AS
    SELECT *
    FROM read_parquet('{caminho_clientes}')
""")

conexao.execute(f"""
    CREATE OR REPLACE TABLE bronze_transactions AS
    SELECT *
    FROM read_parquet('{caminho_transacoes}')
""")

contagem_bronze = conexao.execute("""
    SELECT
        (SELECT COUNT(*) FROM bronze_customers)
            AS clientes,
        (SELECT COUNT(*) FROM bronze_transactions)
            AS transacoes
""").df()

contagem_bronze

,clientes,transacoes
0,9993,100000


In [3]:
validacao_chave_clientes = conexao.execute("""
    SELECT
        COUNT(*) AS total_clientes,
        COUNT(DISTINCT customer_id) AS ids_unicos,
        COUNT(*) - COUNT(DISTINCT customer_id)
            AS duplicidades_excedentes
    FROM bronze_customers
""").df()

validacao_chave_clientes

,total_clientes,ids_unicos,duplicidades_excedentes
0,9993,9993,0


In [4]:
resumo_orfaos = conexao.execute("""
    SELECT
        COUNT(*) AS transacoes_orfas,
        COUNT(DISTINCT t.customer_id) AS clientes_ausentes
    FROM bronze_transactions t
    LEFT JOIN bronze_customers c
        ON t.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""").df()

resumo_orfaos

,transacoes_orfas,clientes_ausentes
0,0,0


In [5]:
clientes_ausentes = conexao.execute("""
    SELECT
        t.customer_id,
        COUNT(*) AS quantidade_transacoes
    FROM bronze_transactions t
    LEFT JOIN bronze_customers c
        ON t.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
    GROUP BY t.customer_id
    ORDER BY quantidade_transacoes DESC, t.customer_id
""").df()

clientes_ausentes

,customer_id,quantidade_transacoes


In [6]:
conexao.execute("""
    CREATE OR REPLACE TABLE silver_transactions AS
    SELECT
        t.transaction_id,
        t.customer_id,
        t.amount,
        t.transaction_type,
        t.transaction_timestamp,
        t.status,
        t.risk_score,
        t.is_fraud,
        c.segment,
        c.credit_score,

        YEAR(t.transaction_timestamp) AS year,
        MONTH(t.transaction_timestamp) AS month,
        DAY(t.transaction_timestamp) AS day,
        DAYOFWEEK(t.transaction_timestamp) AS day_of_week,

        CASE
            WHEN t.amount < 100 THEN 'baixo'
            WHEN t.amount < 1000 THEN 'medio'
            ELSE 'alto'
        END AS amount_band

    FROM bronze_transactions t
    INNER JOIN bronze_customers c
        ON t.customer_id = c.customer_id
""")

print("Tabela Silver criada.")

Tabela Silver criada.


In [7]:
validacao_join = conexao.execute("""
    SELECT
        (SELECT COUNT(*) FROM bronze_transactions)
            AS transacoes_bronze,

        (SELECT COUNT(*) FROM silver_transactions)
            AS transacoes_silver,

        (SELECT COUNT(*) FROM bronze_transactions)
        -
        (SELECT COUNT(*) FROM silver_transactions)
            AS diferenca,

        (SELECT COUNT(*) FROM silver_transactions
         WHERE segment IS NULL)
            AS linhas_sem_segmento,

        (SELECT COUNT(*) - COUNT(DISTINCT transaction_id)
         FROM silver_transactions)
            AS ids_duplicados
""").df()

validacao_join

,transacoes_bronze,transacoes_silver,diferenca,linhas_sem_segmento,ids_duplicados
0,100000,100000,0,0,0


In [8]:
amostra_enriquecimento = conexao.execute("""
    SELECT
        transaction_id,
        amount,
        amount_band,
        transaction_timestamp,
        year,
        month,
        day,
        day_of_week,
        segment,
        credit_score
    FROM silver_transactions
    ORDER BY transaction_id
    LIMIT 10
""").df()

amostra_enriquecimento

,transaction_id,amount,amount_band,transaction_timestamp,year,month,day,day_of_week,segment,credit_score
0,1,331.964562,medio,2023-10-20,2023,10,20,5,Standard,790
1,2,822.086487,medio,2023-04-22,2023,4,22,6,Standard,523
2,3,93.935261,baixo,2023-03-28,2023,3,28,2,High-Risk,481
3,4,40.210678,baixo,2023-01-20,2023,1,20,5,Premium,604
4,5,67.061196,baixo,2024-06-23,2024,6,23,0,Premium,900
5,6,17.848845,baixo,2023-07-28,2023,7,28,5,Premium,622
6,7,140.573782,medio,2024-03-18,2024,3,18,1,Premium,900
7,8,48.962330,baixo,2023-01-24,2023,1,24,2,High-Risk,561
8,9,10.000000,baixo,2024-12-06,2024,12,6,5,High-Risk,482
9,10,13.309878,baixo,2023-06-24,2023,6,24,6,Premium,900


In [9]:
validacao_faixas = conexao.execute("""
    SELECT
        amount_band,
        COUNT(*) AS transacoes,
        MIN(amount) AS menor_valor,
        MAX(amount) AS maior_valor
    FROM silver_transactions
    GROUP BY amount_band
    ORDER BY
        CASE amount_band
            WHEN 'baixo' THEN 1
            WHEN 'medio' THEN 2
            WHEN 'alto' THEN 3
        END
""").df()

validacao_faixas

,amount_band,transacoes,menor_valor,maior_valor
0,baixo,53729,10.000000,99.994114
1,medio,44031,100.002703,999.814267
2,alto,2240,1000.087557,22927.023010


In [10]:
fraude_por_segmento = conexao.execute("""
    SELECT
        segment,
        COUNT(*) AS transacoes,
        COUNT(DISTINCT customer_id) AS clientes,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            AS fraudes,
        ROUND(
            100.0
            * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct
    FROM silver_transactions
    GROUP BY segment
    ORDER BY taxa_fraude_pct DESC
""").df()

fraude_por_segmento

,segment,transacoes,clientes,fraudes,taxa_fraude_pct
0,High-Risk,9155,876,705.0,7.70
1,Standard,29689,2665,655.0,2.21
2,Premium,61156,5563,473.0,0.77


In [11]:
if arquivo_silver.exists():
    arquivo_silver.unlink()

destino_silver = (
    arquivo_silver.as_posix().replace("'", "''")
)

conexao.execute(f"""
    COPY silver_transactions
    TO '{destino_silver}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Silver gravada em:")
print(arquivo_silver)

Silver gravada em:
C:\BigData\bigdata-curso-gabriel\dados\silver\transactions_enriched.parquet


In [12]:
validacao_arquivo_silver = conexao.execute(f"""
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT transaction_id) AS ids_unicos,
        COUNT(*) - COUNT(DISTINCT transaction_id)
            AS ids_duplicados,
        COUNT(*) FILTER (WHERE segment IS NULL)
            AS linhas_sem_segmento
    FROM read_parquet('{destino_silver}')
""").df()

validacao_arquivo_silver

,total_linhas,ids_unicos,ids_duplicados,linhas_sem_segmento
0,100000,100000,0,0


In [13]:
schema_silver = conexao.execute("""
    DESCRIBE silver_transactions
""").df()

schema_silver[["column_name", "column_type"]]

,column_name,column_type
0,transaction_id,BIGINT
1,customer_id,BIGINT
2,amount,DOUBLE
3,transaction_type,VARCHAR
4,transaction_timestamp,TIMESTAMP
5,status,VARCHAR
6,risk_score,DOUBLE
7,is_fraud,BOOLEAN
8,segment,VARCHAR
9,credit_score,INTEGER


In [14]:
conexao.close()

print("Conexão encerrada.")
print("Lab 6 executado com sucesso.")

Conexão encerrada.
Lab 6 executado com sucesso.


## Conclusão

A camada Silver foi construída por meio da integração entre as transações e o cadastro de clientes. O JOIN acrescentou a cada transação o segmento e o credit score do respectivo cliente.

Antes da integração, foi verificada a unicidade de `customer_id` na dimensão de clientes, evitando a multiplicação indevida de registros. Também foram pesquisadas transações sem cliente correspondente, permitindo mensurar eventual perda causada pelo INNER JOIN.

Foram derivadas as colunas de ano, mês, dia, dia da semana e faixa de valor. Essas variáveis facilitam agregações e análises sem que as mesmas transformações precisem ser repetidas nas etapas seguintes.

Após o enriquecimento, foram validadas as quantidades de registros, a ausência de identificadores duplicados e a presença do segmento. A tabela resultante foi persistida em formato Parquet para alimentar as camadas Gold e as análises posteriores.